In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import sklearn.preprocessing
from sklearn.preprocessing import LabelEncoder
import json

In [12]:
# Load data for all sales, calendar and prices
# The sales_train_evaluation.csv file contains the whole range of days
# This will be split into training and evaluation datasets - with only the last 4 weeks being used for evaluation
df_sales = pd.read_csv('../m5-dataset/sales_train_evaluation.csv')
df_calendar = pd.read_csv('../m5-dataset/calendar.csv')
df_prices = pd.read_csv('../m5-dataset/sell_prices.csv')

In [13]:
# reduce the set to reduce notebook computation time
# sample_items = df_sales['item_id'].unique()[:50]
# df_sales = df_sales[df_sales['item_id'].isin(sample_items)]

In [14]:
# Merge together df_sales and df_calendar
df_melted = df_sales.melt(
    id_vars=['item_id', 'store_id'],
    var_name='d',
    value_name='units_sold'
)
df_temp = df_melted.merge(df_calendar, on='d')
df_merged = df_temp.merge(df_prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
# some items do not exist at the start, and some get taken out of inventory, meaning many NaN values
# we fix this by forward and backward filling (not ideal, but this way we avoid losing massive amounts of data)
df_merged['sell_price'] = df_merged.groupby(['item_id', 'store_id'])['sell_price'].ffill().bfill()

#add state column (which wont get converted to category)
df_merged['state'] = df_merged['store_id'].str[:2]

# Drop metadata to reduce file size
df_merged = df_merged.drop(columns=['d', 'month', 'year', 'weekday'])

print(f"df_merged memory usage before cleanup: {round((df_merged.memory_usage(deep=True).sum() / 1024**2), 2)} MB")

# Convert data types to reduce dataset size
df_merged['units_sold'] = pd.to_numeric(df_merged['units_sold'], errors='coerce').astype('int16')
df_merged['wm_yr_wk'] = df_merged['wm_yr_wk'].values.astype(np.int16)
df_merged['wday'] = df_merged['wday'].values.astype(np.int8)
df_merged['snap_CA'] = df_merged['snap_CA'].values.astype(np.int8)
df_merged['snap_TX'] = df_merged['snap_TX'].values.astype(np.int8)
df_merged['snap_WI'] = df_merged['snap_WI'].values.astype(np.int8)

# Convert to categories to further reduce size
for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
    df_merged[col] = df_merged[col].fillna('x') # x means none - placeholder to remove NaN values. In this case - no event

cat_cols = ['item_id', 'store_id', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']
df_merged[cat_cols] = df_merged[cat_cols].astype('category')

print(f"df_merged memory usage after cleanup: {round((df_merged.memory_usage(deep=True).sum() / 1024**2), 2)} MB")

df_merged memory usage before cleanup: 26630.63 MB
df_merged memory usage after cleanup: 8409.75 MB


In [15]:
df = df_merged
df['units_sold'] = pd.to_numeric(df['units_sold'], errors='coerce').astype('int16')
df['wm_yr_wk'] = df['wm_yr_wk'].values.astype(np.int16)
df['wday'] = df['wday'].values.astype(np.int8)
df['snap_CA'] = df['snap_CA'].values.astype(np.int8)
df['snap_TX'] = df['snap_TX'].values.astype(np.int8)
df['snap_WI'] = df['snap_WI'].values.astype(np.int8)

df = df.drop(columns=['wm_yr_wk', 'wday'])
df['date'] = pd.to_datetime(df['date'])

In [16]:
# Extract categories from item_id
df['item_category'] = df['item_id'].str.split('_').str[0]

encoders = {}
le = LabelEncoder()
for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'item_id', 'store_id', 'item_category']:
    df[col] = le.fit_transform(df[col])
    encoders[col] = dict(enumerate(le.classes_))

# for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
#     df[col] = le.fit_transform(df[col])

# df['item_id'] = le.fit_transform(df['item_id'])
# df['store_id'] = le.fit_transform(df['store_id'])

df['day_of_week'] = df['date'].dt.dayofweek
df['day_of_month'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

with open('../Data/encoders.json', 'w') as f:
    json.dump({k: {str(i): v for i, v in d.items()} for k, d in encoders.items()}, f)

In [18]:
df_cpi = pd.read_csv('../Data/Derived_data/CPI_ALL.csv')
df_cpi['year'] = df_cpi['date']
df_cpi = df_cpi[['year', 'cpi_WI_avg', 'cpi_TX_avg', 'cpi_CA_avg']]

df = df.merge(df_cpi, on='year', how='left')
df['state_cpi'] = np.where(df['state'] == 'CA', df['cpi_CA_avg'],
                  np.where(df['state'] == 'TX', df['cpi_TX_avg'],
                           df['cpi_WI_avg']))
df = df.drop(columns=['cpi_CA_avg', 'cpi_TX_avg', 'cpi_WI_avg'])
df['state_cpi_mom_delta'] = df.groupby('store_id')['state_cpi'].pct_change(periods=30)

In [19]:
df_ur = pd.read_csv('../Data/Derived_data/UR_ALL.csv')
df_ur = df_ur[['date', 'ur_WI', 'ur_TX', 'ur_CA']]
df_ur['date'] = pd.to_datetime(df_ur['date'])

df_ur['month'] = df_ur['date'].dt.month
df_ur['year'] = df_ur['date'].dt.year
df_ur = df_ur.drop(columns=['date'])

df = df.merge(df_ur, on=['month', 'year'], how='left')
df['state_ur'] = np.where(df['state'] == 'CA', df['ur_CA'],
                 np.where(df['state'] == 'TX', df['ur_TX'],
                           df['ur_WI']))
df = df.drop(columns=['ur_CA', 'ur_TX', 'ur_WI'])
df['state_ur_mom_delta']  = df.groupby('store_id')['state_ur'].pct_change(periods=30)

In [20]:
df_gas = pd.read_csv('../Data/Derived_data/GAS_ALL.csv')
df_gas = df_gas[['date', 'gas_WI', 'gas_TX', 'gas_CA']]
df_gas['date'] = pd.to_datetime(df_gas['date'])

df = df.sort_values('date')
df_gas = df_gas.sort_values('date')
df = pd.merge_asof(df, df_gas, on='date', direction='backward')

df['state_gas_price'] = np.where(df['state'] == 'CA', df['gas_CA'],
                        np.where(df['state'] == 'TX', df['gas_TX'],
                                 df['gas_WI']))
df = df.drop(columns=['gas_CA', 'gas_TX', 'gas_WI'])
df['state_gas_wow_delta'] = df.groupby('store_id')['state_gas_price'].pct_change(periods=7)
df = df.sort_index()

In [21]:
df_sentiment = pd.read_csv('../Data/Economic_data/Consumer_sentiment.csv')
df_sentiment['date'] = pd.to_datetime(df_sentiment['date'])

df_sentiment['month'] = df_sentiment['date'].dt.month
df_sentiment['year'] = df_sentiment['date'].dt.year
df_sentiment = df_sentiment.drop(columns=['date'])

df = df.merge(df_sentiment, on=['month', 'year'], how='left')

In [22]:
df['state_snap'] = np.where(df['state'] == 'CA', df['snap_CA'],
                   np.where(df['state'] == 'TX', df['snap_TX'],
                            df['snap_WI']))
df = df.drop(columns=['snap_CA', 'snap_TX', 'snap_WI'])

In [23]:
# add tax season - which is usually in february-march, months 2 and 3
# does tax refund increase sales of affect them in any other way? 
# Morgan Stanley paper says it does, increases sales by about 2-3%
# NOTE! Don't really need to add it as a feature in the dataset.
# It would also work to just compare results of the trained model for months 2 and 3

In [24]:
print('Size BEFORE casting as other types and rounding: ')
print(round((df.memory_usage(deep=True).sum() / 1024**2), 2), ' MB')

Size BEFORE casting as other types and rounding: 
10046.23  MB


In [25]:
cols_to_cast_to_int32 = ['item_id', 'store_id', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']
cols_to_cast_to_int16 = ['day_of_week', 'day_of_month', 'month', 'year', 'item_category']
cols_to_cast_to_float16 = ['sell_price', 'state_cpi', 'state_ur', 'state_gas_price', 'consumer_sentiment', 'state_cpi_mom_delta', 'state_ur_mom_delta', 'state_gas_wow_delta']

for col_int32 in cols_to_cast_to_int32:
    df[col_int32] = df[col_int32].values.astype(np.int32)

for col_int16 in cols_to_cast_to_int16:
    df[col_int16] = df[col_int16].values.astype(np.int16)

for col_float16 in cols_to_cast_to_float16:
    df[col_float16] = df[col_float16].values.astype(np.float16)

In [26]:
print('Size AFTER casting as other types and rounding: ')
print(round((df.memory_usage(deep=True).sum() / 1024**2), 2), ' MB')

Size AFTER casting as other types and rounding: 
6772.74  MB


In [27]:
df.head(3)

,item_id,store_id,units_sold,date,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,state,...,month,year,state_cpi,state_cpi_mom_delta,state_ur,state_ur_mom_delta,state_gas_price,state_gas_wow_delta,consumer_sentiment,state_snap
0,1437,0,0,2011-01-29,30,4,4,2,0.459961,CA,...,1,2011,237.125,NaN,12.101562,NaN,3.292969,NaN,74.1875,0
1,428,6,0,2011-01-29,30,4,4,2,1.879883,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0
2,427,6,0,2011-01-29,30,4,4,2,1.879883,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0


In [ ]:
df.to_parquet('../data/dataset_final_all_features.pqt', index=False)

In [ ]:
df_pqt = pd.read_parquet('../data/dataset_final_all_features.pqt')

In [30]:
df_pqt.head()

,item_id,store_id,units_sold,date,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,state,...,month,year,state_cpi,state_cpi_mom_delta,state_ur,state_ur_mom_delta,state_gas_price,state_gas_wow_delta,consumer_sentiment,state_snap
0,1437,0,0,2011-01-29,30,4,4,2,0.459961,CA,...,1,2011,237.125,NaN,12.101562,NaN,3.292969,NaN,74.1875,0
1,428,6,0,2011-01-29,30,4,4,2,1.879883,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0
2,427,6,0,2011-01-29,30,4,4,2,1.879883,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0
3,426,6,5,2011-01-29,30,4,4,2,3.779297,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0
4,425,6,1,2011-01-29,30,4,4,2,2.419922,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0


In [31]:
df_pqt = df_pqt.drop(columns=['state', 'date'])

In [32]:
df_pqt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59181090 entries, 0 to 59181089
Data columns (total 21 columns):
 #   Column               Dtype  
---  ------               -----  
 0   item_id              int32  
 1   store_id             int32  
 2   units_sold           int16  
 3   event_name_1         int32  
 4   event_type_1         int32  
 5   event_name_2         int32  
 6   event_type_2         int32  
 7   sell_price           float16
 8   item_category        int16  
 9   day_of_week          int16  
 10  day_of_month         int16  
 11  month                int16  
 12  year                 int16  
 13  state_cpi            float16
 14  state_cpi_mom_delta  float16
 15  state_ur             float16
 16  state_ur_mom_delta   float16
 17  state_gas_price      float16
 18  state_gas_wow_delta  float16
 19  consumer_sentiment   float16
 20  state_snap           int8   
dtypes: float16(8), int16(6), int32(6), int8(1)
memory usage: 2.9 GB


In [ ]:
df_pqt.to_parquet('../data/dataset_final.pqt', index=False)

In [ ]:
df_pqt_nodate = pd.read_parquet('../data/dataset_final.pqt')

test_items = df_pqt_nodate['item_id'].unique()[:100]
df_test_items = df_pqt_nodate[df_pqt_nodate['item_id'].isin(test_items)]

In [35]:
df_test_items.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1941000 entries, 0 to 59180135
Data columns (total 21 columns):
 #   Column               Dtype  
---  ------               -----  
 0   item_id              int32  
 1   store_id             int32  
 2   units_sold           int16  
 3   event_name_1         int32  
 4   event_type_1         int32  
 5   event_name_2         int32  
 6   event_type_2         int32  
 7   sell_price           float16
 8   item_category        int16  
 9   day_of_week          int16  
 10  day_of_month         int16  
 11  month                int16  
 12  year                 int16  
 13  state_cpi            float16
 14  state_cpi_mom_delta  float16
 15  state_ur             float16
 16  state_ur_mom_delta   float16
 17  state_gas_price      float16
 18  state_gas_wow_delta  float16
 19  consumer_sentiment   float16
 20  state_snap           int8   
dtypes: float16(8), int16(6), int32(6), int8(1)
memory usage: 112.9 MB


In [ ]:
df_for_training = pd.read_parquet('../data/dataset_final.pqt')
df.head()

,item_id,store_id,units_sold,date,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,state,...,month,year,state_cpi,state_cpi_mom_delta,state_ur,state_ur_mom_delta,state_gas_price,state_gas_wow_delta,consumer_sentiment,state_snap
0,1437,0,0,2011-01-29,30,4,4,2,0.459961,CA,...,1,2011,237.125,NaN,12.101562,NaN,3.292969,NaN,74.1875,0
1,428,6,0,2011-01-29,30,4,4,2,1.879883,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0
2,427,6,0,2011-01-29,30,4,4,2,1.879883,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0
3,426,6,5,2011-01-29,30,4,4,2,3.779297,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0
4,425,6,1,2011-01-29,30,4,4,2,2.419922,TX,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,NaN,74.1875,0


In [41]:
# Since the dataset now contains NaN values, replace them with 0
# ALSO - MOVE THIS UP BEFORE COMMITTING
df_for_training['state_cpi_mom_delta']  = df_for_training['state_cpi_mom_delta'].fillna(0)
df_for_training['state_ur_mom_delta']   = df_for_training['state_ur_mom_delta'].fillna(0)
df_for_training['state_gas_wow_delta']  = df_for_training['state_gas_wow_delta'].fillna(0)

In [42]:
df_for_training.head()

,item_id,store_id,units_sold,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,item_category,day_of_week,...,month,year,state_cpi,state_cpi_mom_delta,state_ur,state_ur_mom_delta,state_gas_price,state_gas_wow_delta,consumer_sentiment,state_snap
0,1437,0,0,30,4,4,2,0.459961,1,5,...,1,2011,237.125,0.0,12.101562,0.0,3.292969,0.0,74.1875,0
1,428,6,0,30,4,4,2,1.879883,0,5,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,0.0,74.1875,0
2,427,6,0,30,4,4,2,1.879883,0,5,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,0.0,74.1875,0
3,426,6,5,30,4,4,2,3.779297,0,5,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,0.0,74.1875,0
4,425,6,1,30,4,4,2,2.419922,0,5,...,1,2011,200.250,0.0,8.203125,0.0,2.974609,0.0,74.1875,0


In [ ]:
df_for_training.to_parquet('../data/dataset_final_reduced.pqt', index=False)